# Flow Control and Grids

## What you will build

You will build a folded {{< glossary Timeline >}}, describe its played order
as a {{< glossary FlowMap >}}, and convert positions between the page and the
performance. You will read the flow control a score loader writes, play it,
and check the result. You will also build a {{< glossary BeatGrid >}} whose
measures and beats can be queried in either direction, survive a tempo
change, and be exported for annotation software.

## Before you start

Complete [Nesting and Timestamps](tut02_nesting_and_timestamps.ipynb),
[Loading Real Data](tut04_loading_data.ipynb), and
[Timeline Groups](tut05_timeline_groups.ipynb) first. This notebook follows
[Alignment Bundles](tut06_alignment_bundles.ipynb) in reading order but does
not depend on its bundles or claims.

In [1]:
import warnings
from fractions import Fraction
from pathlib import Path
from tempfile import TemporaryDirectory

from timetoalign import (
    BeatGrid,
    BeatGridSegment,
    BeatPolicy,
    ContinuousLogicalTimeline,
    Ms3Loader,
)
from timetoalign.testdata import ensure_data
from timetoalign.timelines import FlowMap
from timetoalign.timelines.flow import measure_ranges

score_data = ensure_data("score")

## A page is not a performance

A repeat sign or *Da Capo* changes traversal order without changing the page.
The written timeline is **folded**; the route actually played is **unfolded**.

In [2]:
written_score = ContinuousLogicalTimeline.from_events(
    [
        {
            "id": "written_end",
            "temporal_type": "instant",
            "event_type": "Boundary",
            "instant": Fraction(12),
        }
    ],
    uid="written_score",
    name="Three written spans",
)
span_bounds = {
    "A": (Fraction(0), Fraction(4)),
    "B": (Fraction(4), Fraction(8)),
    "C": (Fraction(8), Fraction(12)),
}
written_order = list(span_bounds)
played_order = ["A", "A", "C"]
page_and_performance = {
    "written length": written_score.length,
    "written order": written_order,
    "played order": played_order,
}
page_and_performance

{'written length': Coordinate(Fraction(12, 1), quarters),
 'written order': ['A', 'B', 'C'],
 'played order': ['A', 'A', 'C']}

The length is a {{< glossary Coordinate >}} measured in quarters. The page
contains A, B, and C once each, but the performance repeats A and skips B.

## Naming the played spans

A {{< glossary Region >}} gives a stable id to a span of the written score.
We reuse `create_region` from the nesting tutorial, giving each span its letter
as its id.

In [3]:
created_regions = [
    written_score.create_region(start, end, id=span_id)
    for span_id, (start, end) in span_bounds.items()
]
opening_region = created_regions[0]
closing_region = created_regions[2]
named_spans = written_score.list_regions()
region_overview = {
    "opening span": opening_region,
    "region ids": named_spans,
}
region_overview

{'opening span': Region('A', 0-4 quarters), 'region ids': ['A', 'B', 'C']}

Each region keeps its quarter-valued start and end. These ids now let the
played route refer to legible musical spans rather than unexplained numbers.

## Building the map

A {{< glossary FlowMap >}} lists the spans of one timeline in the order they
are played. `create_flow_map(chunks)` takes them as region ids or Region
objects, builds the map, attaches it to the timeline, and returns it. A
FlowMap you build yourself joins a timeline with `add_flow_map`.

In [4]:
flow_chunks = [played_order[0], opening_region, closing_region]
performance_map = written_score.create_flow_map(flow_chunks, id="performance")
written_score.add_flow_map(FlowMap("written_score", written_order, id="as_written"))
retrieved_map = written_score.get_flow_map("performance")
available_maps = written_score.list_flow_maps()
map_overview = {
    "selected map": retrieved_map,
    "its chunks": retrieved_map.chunks,
    "attached ids": available_maps,
}
map_overview

{'selected map': FlowMap('performance', timeline='written_score', 3 chunks),
 'its chunks': ('A', 'A', 'C'),
 'attached ids': ['performance', 'as_written']}

The `flow_chunks` argument mixes a region id with Region objects; the map
holds them all as ids. `get_flow_map` retrieves an attached map by its id and
`list_flow_maps` enumerates them.

## Unfolding one position

`resolve()` reads a FlowMap against its timeline: each span becomes a stretch
of the page, placed where the one before it ends in the performance. Its
`unfold(coord)` takes one folded position and returns a list, because
repeated material can occur at several positions in performance.

In [5]:
performance = performance_map.resolve(written_score)
repeated_folded = written_score.get_coordinate(Fraction(1))
skipped_folded = written_score.get_coordinate(Fraction(5))
repeated_unfolded = performance.unfold(repeated_folded)
skipped_unfolded = performance.unfold(skipped_folded)
unfold_report = {
    "folded position in A": repeated_folded,
    "played positions": repeated_unfolded,
    "folded position in B": skipped_folded,
    "played positions for skipped B": skipped_unfolded,
}
unfold_report

{'folded position in A': IdCoordinate(Fraction(1, 1), quarters, 'written_score'),
 'played positions': [Coordinate(Fraction(1, 1), quarters),
  Coordinate(Fraction(5, 1), quarters)],
 'folded position in B': IdCoordinate(Fraction(5, 1), quarters, 'written_score'),
 'played positions for skipped B': []}

The position in A appears once in each rendition, at 1 and 5 quarters. The
empty list for B is the correct answer, not a failure: that written span is
absent from this route. The played positions are Coordinates in the written
timeline's unit, quarters.

## Folding back

`fold(coord)` goes in the other direction: many possible performance
positions collapse to one position on the written page.

In [6]:
later_unfolded = repeated_unfolded[1]
folded_position = performance.fold(later_unfolded)
fold_report = {
    "later performance position": later_unfolded,
    "position on the page": folded_position,
    "matches the original value": folded_position.value == repeated_folded.value,
}
fold_report

{'later performance position': Coordinate(Fraction(5, 1), quarters),
 'position on the page': Coordinate(Fraction(1, 1), quarters),
 'matches the original value': True}

The second rendition folds back to the original written value. Keep the
directions distinct: unfolding is one-to-many; folding is many-to-one.

## The performance as a timeline of its own

A virtual timeline states that a FlowMap was applied to its timeline. It has
an id, the written timeline's unit, and the performance's length, and holds
nothing else: no events and no children. Positions convert through it in
both directions, given the timeline it comes from.

In [7]:
performed = written_score.create_virtual_timeline(performance_map, id="performed")
performed_overview = {
    "virtual timeline": performed,
    "played positions": performed.unfold(repeated_folded, written_score),
    "virtual timelines": written_score.list_virtual_timelines(),
}
performed_overview

{'virtual timeline': VirtualTimeline('performed', map='performance', timeline='written_score', length=12 quarters),
 'played positions': [Coordinate(Fraction(1, 1), quarters),
  Coordinate(Fraction(5, 1), quarters)],
 'virtual timelines': ['performed']}

The performance is twelve quarters long, as the page is: A twice and C once
take as long as A, B and C.

## Inspecting repeat signs

A score's measure table records its repeat signs. Inspecting those rows first
makes the flow control a loader writes from them easier to read.

In [8]:
score_dir = score_data / "beethoven_woo71"
score_loader = Ms3Loader.from_file(
    score_dir / "WoO71.notes.tsv",
    score_dir / "WoO71.measures.tsv",
)
measure_data = score_loader.store.measures
measure_table = measure_data.to_dataframe()
repeat_mask = measure_table["start_repeat"] | measure_table["end_repeat"]
repeat_columns = ["mc", "mn", "start_repeat", "end_repeat", "next"]
repeat_rows = measure_table.loc[
    repeat_mask,
    repeat_columns,
]
repeat_events = repeat_rows.head(6)
repeat_events

,mc,mn,start_repeat,end_repeat,next
11,12,10,True,False,13
20,21,19,False,True,"12, 22"
32,33,29,True,False,34
41,42,38,False,True,"33, 43"
83,84,76,True,False,85
92,93,85,False,True,"84, 94"


These six rows form three repeat-start and repeat-end pairs. The `next`
column shows the encoded destinations; each repeat end has both a backward
destination and the following written measure.

## Ids and names of flow-control elements

A loader writes these signs as the timeline's **flow control**: a list of
elements of five kinds. A jump is where the flow may leave the axis, with one
destination for every place it may go on at; a jump target is a place a
destination names (a repeat start, a segno, a coda, a bracket); a break is a
place the flow does not go on across (a *Fine*, the end of a first bracket);
a section break ends a section; a barline is a barline the source prints.

Every element of a timeline (a region, a jump, a jump target, a break) has an
`id` that identifies it, unique across everything you save together in one
manifest, and may have a name for display. A loader gives ids where the
source has none: one prefix per kind followed by a random suffix, `jump_…`,
`jump_target_…`, `break_…`, `section_break_…`, `barline_…`. A short score
with a *Fine*, a segno and a coda shows them.

In [9]:
flow_dir = score_data / "flow_control" / "flow_only"
with warnings.catch_warnings(record=True) as flow_only_warnings:
    warnings.simplefilter("always")
    flow_score = Ms3Loader.from_file(
        flow_dir / "out_of_the_flow_experience-flow_only.measures.tsv",
    ).create_timeline(uid="flow_only")
flow_elements = flow_score.flow_control
element_ids = {
    "regions": {region.id: region.name for region in written_score.iter_regions()},
    "breaks": {brk.id: brk.name for brk in flow_elements.breaks},
    "jump targets": {target.id: target.name for target in flow_elements.jump_targets},
    "jump destinations": {
        jump.id: [destination.to for destination in jump.targets]
        for jump in flow_elements.jumps
    },
}
element_ids

{'regions': {'A': None, 'B': None, 'C': None},
 'breaks': {'break_1r2atzexpoczi': None,
  'break_33vutn8uttd0e': None,
  'break_364zz5zo2kom6': 'fine',
  'break_1n8dksx4wjliv': None},
 'jump targets': {'jump_target_2qtqsrqposh4o': None,
  'jump_target_3kajzabnk37ik': None,
  'jump_target_0mkxttxn9vnfk': None,
  'jump_target_0izxt9nz29uy7': None,
  'jump_target_2f939489wxt1i': 'segno',
  'jump_target_2mwicbtuj64op': None,
  'jump_target_1jiz9ha8f5wx0': 'codab',
  'jump_target_0deg2knggru4t': None},
 'jump destinations': {'jump_37934feiyv3sf': [None],
  'jump_1q1yxi30ieu2v': [None,
   'jump_target_0mkxttxn9vnfk',
   'jump_target_3kajzabnk37ik'],
  'jump_34jkroql6o0n3': ['jump_target_2qtqsrqposh4o'],
  'jump_1ol3zobob7vrh': [None],
  'jump_08txti49972a1': ['jump_target_1jiz9ha8f5wx0'],
  'jump_25ym40vdw5gzk': ['jump_target_2mwicbtuj64op'],
  'jump_1bu16a13ndfvh': ['jump_target_0izxt9nz29uy7'],
  'jump_3ojqmr8hqz9vn': ['jump_target_2f939489wxt1i'],
  'jump_1oa34haeq5xo3': [None, 'jump_targ

The written score's regions carry the ids you gave and no names; the loader
names the *Fine*, the segno and the coda as the source names them. A
destination that goes on at a jump target names that target's id as its
`to`; a plain repeat and the *Da Capo* name none and go back to the start
of their section, and a bracket the flow simply goes on into states a
distance of 0 instead.

Loading the fragment, the loader also warned that some of its measures do
not add up to their bars. The cell above recorded the warning instead of
letting it print; its message names each measure and how it does not fit:

In [10]:
print(flow_only_warnings[0].message)

Floating measures that do not add up to their bars:
  short: measure '6' (number 2) ends 1 quarter into a bar of 4, and measure '8' (number 4), which it leads to, does not complete the bar;
  short: measure '7' (number 3) ends 2 quarters into a bar of 4, and it leads to no measure of the map;
  doubtful split, not joined: measure '12' (number 8) and measure '13' (number 9) together fill a bar of 4 quarters, but '13' states offset 0, not 2 where '12' ends


## Computing and checking the route

`get_flow()` plays a timeline's flow control and writes nothing. Its result
lists the stretches of the score in the order played and converts positions
like a resolved FlowMap; its `findings` report what looks wrong. Stated in
measures, the route of WoO 71 is easy to compare with the score.

In [11]:
with warnings.catch_warnings(record=True) as woo71_warnings:
    warnings.simplefilter("always")
    score_timeline = score_loader.create_timeline(uid="woo71")
route = score_timeline.get_flow()
played_measures = measure_ranges(
    [stretch.source for stretch in route.chunks], score_timeline.measure_map
)
flow_report = {
    "folded measures": len(score_timeline.measure_map),
    "played measures": sum(stretch.n_measures for stretch in played_measures),
    "first stretches": played_measures[:3],
    "findings": [finding.message for finding in route.findings] or "none",
}
flow_report

{'folded measures': 397,
 'played measures': 505,
 'first stretches': (MeasureRange(start=1, end=22, sections=()),
  MeasureRange(start=12, end=43, sections=()),
  MeasureRange(start=33, end=94, sections=())),
 'findings': 'none'}

The default route visits 505 measures from a 397-measure folded score, in
right-open ranges of measure counts, and finds nothing to report. The
loader's one warning, recorded as for the fragment, is about the measures
themselves: at three places a measure states an offset, or reaches a
length, that does not fit its bar.

In [12]:
print(woo71_warnings[0].message)

Floating measures that do not add up to their bars:
  offset mismatch: measure '43' (number 38) states offset 3/2, not 1 where '42' ends;
  offset mismatch: measure '262' (number 237) states offset 3/2, not 4 where '261' ends;
  overlength: measures '371', '372' (number 346) reach 9/2 quarters into a bar of 3


The short score above is another matter: its second of three brackets has
no sign at its end, so its flow stops there and never plays the rest.

In [13]:
[finding.message for finding in flow_score.get_flow().findings]

["The flow ends at the break 'break_33vutn8uttd0e' at 10 quarters: no jump there has a destination that holds.",
 'The stretch [10, 37) quarters is never played.']

## Segmenting the folded score

`create_flow_maps_from_flow_control()` writes the stretches flow control
never divides, its atomic sections, as Regions named by letters, and the
route as a FlowMap of them. Their starts divide the folded timeline into a
{{< glossary SegmentLine >}} of adjacent, inspectable sections.

In [14]:
computed = score_timeline.create_flow_maps_from_flow_control()
section_starts = [section.start for section in computed.sections]
boundary_coordinates = [*section_starts, score_timeline.length]
score_segments = score_timeline.create_segment_line(boundary_coordinates)
segment_report = {
    "first sections": [section.name for section in computed.sections[:4]],
    "first boundaries": boundary_coordinates[:4],
    "boundary count": len(boundary_coordinates),
    "segment count": score_segments.n_children,
    "route FlowMap": computed.flow_maps[0],
}
segment_report

{'first sections': ['A', 'B', 'C', 'D'],
 'first boundaries': [Coordinate(Fraction(0, 1), quarters),
  Coordinate(Fraction(20, 1), quarters),
  Coordinate(Fraction(38, 1), quarters),
  Coordinate(Fraction(58, 1), quarters)],
 'boundary count': 27,
 'segment count': 26,
 'route FlowMap': FlowMap('flow_0ly28qh2uw3cw', timeline='woo71', 37 chunks, convention='default')}

The boundary coordinates retain their quarter unit. Twenty-seven boundaries
divide the score into twenty-six adjacent segments, one per atomic section,
and the SegmentLine keeps those segments as children of one timeline. The
route's FlowMap lists the sections in the order played.

## Beats as a queryable grid

A {{< glossary BeatGrid >}} answers where beats and measures fall in a
recording. It is not a timeline: it stores one tempo statement — the anchor
instant, the tempo, the meter, and which beat of a bar the anchor is — and
generates every beat from it. `extent` bounds the grid, normally with the
length of the audio.

In [15]:
grid = BeatGrid.from_tempo(120, metro="3/4", start=0, extent=12)
grid_overview = {
    "the grid": grid,
    "one stated segment": grid.segments[0],
    "beat length in seconds": grid.segments[0].beat_seconds,
    "measures generated": grid.n_measures,
}
grid_overview

{'the grid': BeatGrid(1 segment, extent=12, 8 measures),
 'one stated segment': BeatGridSegment(start=0, end=12, bpm=120, policy=3/4, battito=1),
 'beat length in seconds': Fraction(1, 2),
 'measures generated': 8}

At 120 BPM a beat lasts half a second, so a 3/4 bar lasts one and a half and
twelve seconds hold eight measures. Nothing was stored per beat to say so.

## Both directions

`seconds_at(measure, beat)` turns a label into a
{{< glossary Coordinate >}} in seconds; `position_at(seconds)` turns a
position into the beat sounding there.

In [16]:
labelled_instant = grid.seconds_at(3, 2)
sounding_beat = grid.position_at(6.25)
direction_report = {
    "measure 3, beat 2": labelled_instant,
    "beat sounding at 6.25 seconds": sounding_beat,
    "that beat's own instant": sounding_beat.instant,
    "its measure": sounding_beat.measure,
    "it opens a measure": sounding_beat.is_downbeat,
    "quarters in the whole grid": grid.quarters_between(0, 12),
}
direction_report

{'measure 3, beat 2': Coordinate(3.5, seconds),
 'beat sounding at 6.25 seconds': GridBeat(seconds=Fraction(6, 1), segment=0, measure=5, segment_measure=5, beat=1),
 "that beat's own instant": Coordinate(6.0, seconds),
 'its measure': 5,
 'it opens a measure': True,
 'quarters in the whole grid': Fraction(24, 1)}

`position_at` names the last beat at or before the position, so a query
between two beats answers with the earlier one — 6.25 seconds sounds within
the downbeat of measure 5. `quarters_between` measures notated length rather
than clock time.

A beat carries its position twice, for two jobs. `seconds` is the exact ratio
the grid computed with, which is what the beat's display leads with; `instant`
is that position as a seconds {{< glossary Coordinate >}} — the same value
`seconds_at` returns for the beat's own label, so the two directions meet.

## The beat table and its exports

`get_beat_table()` renders every beat the grid states. It is the only table
the grid builds, and the annotation-file exports are renderings of it.
Downbeats are the rows whose `beat` is 1.

In [17]:
beat_table = grid.get_beat_table()
downbeat_rows = beat_table[beat_table["beat"] == 1]
with TemporaryDirectory() as temporary_dir:
    export_path = Path(temporary_dir) / "beats.csv"
    exported_rows = grid.export_to_csv(
        export_path,
        format="sonic_visualiser",
    )
    exported_text = export_path.read_text(encoding="utf-8")
    first_export_lines = exported_text.splitlines()[:4]
table_report = {
    "first beats": beat_table.head(4),
    "beats": len(beat_table),
    "downbeats": len(downbeat_rows),
    "rows written": exported_rows,
    "first CSV lines": first_export_lines,
}
table_report

{'first beats':    seconds  segment  segment_seconds  measure  beat
 0      0.0        0              0.0        1     1
 1      0.5        0              0.5        1     2
 2      1.0        0              1.0        1     3
 3      1.5        0              1.5        2     1,
 'beats': 24,
 'downbeats': 8,
 'rows written': 24,
 'first CSV lines': ['TIME,LABEL', '0.0,M1B1', '0.5,M1B2', '1.0,M1B3']}

The exported first column is measured in seconds, matching the annotation
file's own convention. The temporary export is genuinely written and read
back; it leaves no tutorial artefact behind.

## When the tempo changes

One segment states one tempo. A recording that changes tempo is several
`BeatGridSegment`s in one grid: you state each segment without an end, and
the grid orders them and bounds each by the next one's start. A
`BeatPolicy` says how the lattice beats group into bars.

In [18]:
three_four = BeatPolicy.uniform(Fraction(1), 3, name="3/4")
opening = BeatGridSegment(start=0, bpm=120, policy=three_four, battito=1)
faster = BeatGridSegment(start=6, bpm=180, policy=three_four, battito=1)
changing = BeatGrid([opening, faster], extent=12)
across_the_change = changing.get_beat_table()
change_report = {
    "bounded segments": changing.segments,
    "measures generated": changing.n_measures,
    "around the change": across_the_change[
        (across_the_change["seconds"] >= 5) & (across_the_change["seconds"] <= 7)
    ],
    "quarters from 5 to 7 seconds": changing.quarters_between(5, 7),
}
change_report

{'bounded segments': (BeatGridSegment(start=0, end=6, bpm=120, policy=3/4, battito=1),
  BeatGridSegment(start=6, end=12, bpm=180, policy=3/4, battito=1)),
 'measures generated': 10,
 'around the change':      seconds  segment  segment_seconds  measure  beat
 10  5.000000        0         5.000000        4     2
 11  5.500000        0         5.500000        4     3
 12  6.000000        1         0.000000        5     1
 13  6.333333        1         0.333333        5     2
 14  6.666667        1         0.666667        5     3
 15  7.000000        1         1.000000        6     1,
 'quarters from 5 to 7 seconds': Fraction(5, 1)}

The grid gave the first segment the second one's start as its end. Beats last
half a second before six seconds and a third of a second after, so the two
seconds around the change hold five quarters — a count no single tempo
produces, because `quarters_between` integrates each segment's own tempo
instead of averaging them.

## Two numberings of one lattice

`numbering="set"` counts measures across the whole grid; `numbering="segment"`
restarts the count at each segment. They label the same beats, so you can read
them side by side.

In [19]:
per_segment = changing.get_beat_table(numbering="segment")
both_numberings = across_the_change.assign(segment_measure=per_segment["measure"])
numbering_report = both_numberings[
    (both_numberings["seconds"] >= 5) & (both_numberings["seconds"] <= 7)
]
numbering_report

,seconds,segment,segment_seconds,measure,beat,segment_measure
10,5.000000,0,5.000000,4,2,4
11,5.500000,0,5.500000,4,3,4
12,6.000000,1,0.000000,5,1,1
13,6.333333,1,0.333333,5,2,1
14,6.666667,1,0.666667,5,3,1
15,7.000000,1,1.000000,6,1,2


The faster segment opens measure 5 of the recording and measure 1 of its own
count. Use whichever the receiving tool expects: a whole-recording annotation
wants the running count, a per-section analysis the restarted one.

## What you learned

- You can distinguish a folded page from its unfolded performance order.
- You can name written spans with regions.
- You can build, attach, retrieve, and list flow maps.
- You can unfold one written position to zero, one, or several played positions.
- You can fold a played position back to one place on the page.
- You can state a performance as a virtual timeline and convert positions through it.
- You can inspect the repeat signs of a score and the flow control a loader writes from them.
- You can compute a score's route and read what playing it found.
- You can write a score's atomic sections and create a segment line from them.
- You can build a beat grid from one tempo statement and bound it with an extent.
- You can turn a measure and beat into an instant, and an instant back into a beat.
- You can render every beat as a table and export a real annotation CSV.
- You can combine several tempo segments into one grid that changes tempo.
- You can read a grid's measures counted across the whole grid or per segment.

## Next

Continue with [The Data Model](tut08_data_model.ipynb).

## Go deeper

- [Flow control](../howto/how01_flow_control.ipynb)
- [Beat grids](../howto/how01_beat_grids.ipynb)